<a href="https://colab.research.google.com/github/jiyanarikan/Ark-Risk-Lab/blob/main/CompVision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install wilds torch torchvision

In [9]:
from wilds import get_dataset

In [ ]:
print("Downloading dataset")

# This downloads to a folder in Colab env
dataset = get_dataset(dataset="poverty", download=True)
train_data = dataset.get_subset("train")
print(f"Total training images loaded: {len(train_data)}")

In [ ]:
# Load ResNet trained on standard images
weights = models.ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=weights)

# Leaves us with a model that outputs a flat vector of numbers
feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
feature_extractor.eval()

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

data_path = '/content/drive/MyDrive/CompVision_Poverty_Research'
if not os.path.exists(data_path):
    os.makedirs(data_path)

print(f"Data saved to: {data_path}")

In [ ]:
# Print the field names without loading the whole array into a DF
print("Fields in metadata:", dataset.metadata_fields)

# Pull the metadata for the first 5 samples manually
for i in range(5):
    _, _, meta = train_data[i]
    print(f"Sample {i} metadata: {meta.tolist()}")

In [ ]:
import torch
import torchvision.models as models
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm

In [ ]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
feature_extractor.to(device)

In [ ]:
# Filter for 500 Urban (Source) and 500 Rural (Target) samples
metadata = train_data.metadata_array
urban_indices = torch.where(metadata[:, 0] == 1)[0][:500].tolist()
rural_indices = torch.where(metadata[:, 0] == 0)[0][:500].tolist()
subset_indices = urban_indices + rural_indices

In [ ]:
loader = DataLoader(torch.utils.data.Subset(train_data, subset_indices), batch_size=32)

all_features = []
all_domains = [] # 1 = Urban, 0 = Rural
all_wealth = []  # Dependent variable (y)

In [ ]:
with torch.no_grad():
    for imgs, labels, metadata in tqdm(loader):
        imgs = imgs[:, :3, :, :].to(device)
        feats = feature_extractor(imgs).view(imgs.size(0), -1)

        all_features.append(feats.cpu().numpy())
        all_domains.append(metadata[:, 0].numpy())
        all_wealth.append(labels.numpy())

In [ ]:
X = np.vstack(all_features)
y_domain = np.concatenate(all_domains)
y_wealth = np.concatenate(all_wealth)

print("\nSuccess! X shape:", X.shape, "| y_domain shape:", y_domain.shape)